# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all Record Set @id's, their names and available fields
record_sets = metadata.record_sets
print("Available Record Sets (by @id):")
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']} | name: {rs.get('name','n/a')}")
    print("  Fields:")
    for field in rs.get('field', []):
        field_object = field if isinstance(field, dict) else metadata.find_by_id(field)
        print(f"    - Field @id: {field_object.get('@id', field if isinstance(field, str) else 'n/a')}, name: {field_object.get('name','n/a')}, dataType: {field_object.get('dataType','n/a')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets into DataFrames
dfs = {}
record_set_ids = [rs['@id'] for rs in metadata.record_sets]
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dfs[record_set_id] = pd.DataFrame(records)
            print(f"Loaded {len(dfs[record_set_id])} records from RecordSet '{record_set_id}'")
        else:
            print(f"No records found for RecordSet '{record_set_id}'.")
    except Exception as e:
        print(f"Failed to load records for RecordSet '{record_set_id}': {e}")

# For demonstration, select the first non-empty RecordSet:
if dfs:
    first_record_set_id = list(dfs.keys())[0]
    print(f"Columns in '{first_record_set_id}':")
    print(dfs[first_record_set_id].columns.tolist())
    dfs[first_record_set_id].head()
else:
    print("No tabular records could be loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Choose numeric field(s) and grouping field(s) for EDA, using field @id
if dfs:
    # Infer likely numeric fields (e.g. containing 'count', 'mw', 'abundance', 'coverage', etc.)
    cols = dfs[first_record_set_id].columns
    possible_num_fields = [c for c in cols if any(s in c.lower() for s in ['count','coverage','abundance','mw','weight','percent','number','int','quantity'])]
    if possible_num_fields:
        numeric_field = possible_num_fields[0] # or prompt user for selection
        print(f"Using field '{numeric_field}' for numeric EDA")
    else:
        numeric_field = cols[0]
        print(f"Defaulting to first column '{numeric_field}' for numeric EDA")
    
    # Filter (threshold is mean if possible)
    try:
        num_vals = pd.to_numeric(dfs[first_record_set_id][numeric_field], errors='coerce')
        threshold = num_vals.mean() if not np.isnan(num_vals.mean()) else 10
        filtered_df = dfs[first_record_set_id][num_vals > threshold].copy()
        print(f"Filtered records where '{numeric_field}' > {threshold:.2f} (mean): {len(filtered_df)} records")
        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (num_vals - num_vals.mean()) / num_vals.std()
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    except Exception as e:
        print(f"Cannot perform numeric filter + normalization: {e}")
    
    # Try grouping by a categorical field (e.g. description, accession, etc.)
    possible_group_fields = [c for c in cols if c != numeric_field]
    group_field = None
    for c in possible_group_fields:
        if any(word in c.lower() for word in ["description", "accession", "category", "group", "type"]):
            group_field = c
            break
    if group_field is None and len(possible_group_fields) > 0:
        group_field = possible_group_fields[0]
    # Try grouping if group_field exists and field is not unique across all rows
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped mean {numeric_field} by field '{group_field}':")
        print(grouped_df.head())
else:
    print("No record set DataFrames available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dfs:
    # Plot histogram and boxplot for the numeric field
    plt.figure(figsize=(14,6))
    num_vals = pd.to_numeric(dfs[first_record_set_id][numeric_field], errors='coerce').dropna()
    plt.subplot(1,2,1)
    sns.histplot(num_vals, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")

    plt.subplot(1,2,2)
    sns.boxplot(x=num_vals)
    plt.title(f"Boxplot of {numeric_field}")
    plt.xlabel(numeric_field)

    plt.tight_layout()
    plt.show()
    
    # If grouped, plot group means
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(10,4))
        grouped_df.head(20).plot(kind='bar')
        plt.title(f"Mean {numeric_field} for each {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(group_field)
        plt.show()
else:
    print("No data for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded and explored the FAIR^2 Croissant dataset for Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. We identified record sets, extracted tabular data via their `@id`, and demonstrated data filtering, normalization, grouping, and plotting by field. These steps offer a reproducible workflow for further biological or statistical analysis using `mlcroissant` and other Python tools.

- Use RecordSet and Field `@id` for consistent, schema-driven processing.
- Adapt filtering, normalization, and grouping as needed for your own hypotheses.
- Visual inspection supports discovery of data characteristics and can inform downstream analysis or modeling workflows.